<a href="https://www.kaggle.com/code/kannammaivr/scikit-learn?scriptVersionId=342343623" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
#from sklearn.model_selection import train_test_split, GridSearchCV
#from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score, classification_report
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingClassifier




# --- Generate synthetic imbalanced data (stand-in for fraud dataset) ---
x, Y = make_classification(
    n_samples=2000, n_features=6, n_informative=4,
    weights=[0.95, 0.1],random_state=21)
clf1=LogisticRegression()
clf1.fit(x,Y)
clf2 = KNeighborsRegressor(n_neighbors=10)
clf2.fit(x, Y)
clf3=VotingClassifier(estimator =[(clf1,clf1),(clf2,clf2)],weight(0.95,0.20))

SyntaxError: '(' was never closed (4025286429.py, line 22)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score, classification_report

# --- Generate synthetic imbalanced data (stand-in for fraud dataset) ---
X, y = make_classification(
    n_samples=2000, n_features=6, n_informative=4,
    weights=[0.95, 0.05], random_state=42
)

# Simulate one skewed feature, e.g. "transaction amount"
# np.log1p compresses the right-skew so extreme values don't dominate (handles zeros safely)
X[:, 0] = np.abs(X[:, 0]) * 1000
X[:, 0] = np.log1p(X[:, 0])

print("Class balance:", np.bincount(y))

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# --- Scaling (fit on train only, transform both) ---
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- sample_weight (per-row importance), built with np.linspace ---
sample_weights = np.linspace(0.5, 2.0, num=len(X_train_scaled))
np.random.shuffle(sample_weights)

# --- Custom scorer (recall on the minority "fraud" class) ---
fraud_recall = make_scorer(recall_score, pos_label=1)

# --- GridSearchCV: tunes C and class_weight, optimizing for fraud recall ---
param_grid = {
    "C": np.linspace(0.01, 2.0, 10),
    "class_weight": ["balanced", None]
}

grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    scoring=fraud_recall,
    cv=5,
    n_jobs=-1,
    refit=True,
    verbose=1
)

# sample_weight passed through fit() — separate from class_weight, which is tuned in param_grid
grid.fit(X_train_scaled, y_train, sample_weight=sample_weights)

print("Best params:", grid.best_params_)
print("Best CV recall score:", grid.best_score_)

# --- Predict and compare against actual ---
y_pred = grid.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=["not fraud", "fraud"]))

# --- Inspect all tried combinations ---
results = pd.DataFrame(grid.cv_results_)
print(results[["param_C", "param_class_weight", "mean_test_score", "rank_test_score"]]
      .sort_values("rank_test_score").head(10))


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

# 1. Load data
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Build pipeline: scale, then classify
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

# 4. Define hyperparameter grid (stepname__paramname)
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__class_weight': [None, 'balanced']
}

# 5. Wrap pipeline in GridSearchCV
grid = GridSearchCV(pipe, param_grid, cv=5)

# 6. Fit — this tries every combination across 5 folds
grid.fit(X_train, y_train)

# 7. Results
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)
print("Test score:", grid.score(X_test, y_test))



         

In [ ]:
#assignment 2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.pipeline import Pipeline

# 1. Create data with a clear outlier
np.random.seed(0)
X = np.concatenate([np.random.normal(loc=50, scale=5, size=200), [500, 520]]).reshape(-1, 1)

# 2. Build a pipeline for each scaler
standard_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

quantile_pipeline = Pipeline(steps=[
    ("scaler", QuantileTransformer(output_distribution="normal", n_quantiles=100, random_state=0))
])

# 3. Fit + transform through each pipeline
X_standard = standard_pipeline.fit_transform(X)
X_quantile = quantile_pipeline.fit_transform(X)

# 4. Plot histograms side by side for comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(X, bins=30, color="gray")
axes[0].set_title("Original Data (with outliers)")

axes[1].hist(X_standard, bins=30, color="steelblue")
axes[1].set_title("After StandardScaler")

axes[2].hist(X_quantile, bins=30, color="darkorange")
axes[2].set_title("After QuantileTransformer")

for ax in axes:
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


In [ ]:
#Assignment1
import numpy as np
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Sample data (swap in your own X, y)
X_train = np.array([[1], [5], [2], [8], [3], [10], [4], [7]], dtype=float)
y_train = np.array([2, 10, 4, 16, 6, 20, 8, 14], dtype=float)
X_test = np.array([[6], [9]], dtype=float)

# ---- Manual approach ----
qt = QuantileTransformer(n_quantiles=8, output_distribution="normal", random_state=0)
X_train_manual = qt.fit_transform(X_train)
model_manual = LinearRegression()
model_manual.fit(X_train_manual, y_train)

X_test_manual = qt.transform(X_test)   # note: transform only, not fit_transform
preds_manual = model_manual.predict(X_test_manual)

# ---- Pipeline approach ----
pipeline = Pipeline([
    ("quantile", QuantileTransformer(n_quantiles=8, output_distribution="normal", random_state=0)),
    ("model", LinearRegression())
])
pipeline.fit(X_train, y_train)
preds_pipeline = pipeline.predict(X_test)

# ---- Compare ----
print("Manual predictions:  ", preds_manual)
print("Pipeline predictions:", preds_pipeline)
print("Match:", np.allclose(preds_manual, preds_pipeline))

# Bonus: confirm the transform step itself matches
print("Transformed X_train equal:",
      np.allclose(X_train_manual, pipeline.named_steps["quantile"].transform(X_train)))

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression

# Sample data with an outlier (800)
x = np.array([2, 4, 6, 8, 16, 87, 800]).reshape(-1, 1)
y = np.array([0, 0, 0, 1, 1, 1, 1])  # dummy labels just to demo the pipeline

# Build the pipeline
pipeline = Pipeline([
    ('scale', QuantileTransformer(n_quantiles=7, output_distribution='normal', random_state=42)),
    ('model', LogisticRegression())
])

pipeline.fit(x, y)
predictions = pipeline.predict(x)

# To visualize what QuantileTransformer did internally, pull it out of the fitted pipeline
transformed_x = pipeline.named_steps['scale'].transform(x)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(range(len(x)), x)
axes[0].set_title("Before QuantileTransformer")
axes[1].scatter(range(len(transformed_x)), transformed_x, color='orange')
axes[1].set_title("After QuantileTransformer")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

arr = np.array(["High","Low","Medium"]).reshape(-1,1)

enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

print(enc.fit_transform(arr))
print(enc.transform(arr))
print(enc.transform(np.array(["new"]).reshape(-1,1)))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

arr =np.array(["High","Low","Medium"]).reshape(-1,1)

enc =OneHotEncoder(sparse_output=False,handle_unknown="ignore")
print(enc.fit_transform(arr))
print(enc.transform(arr))
print(enc.transform(np.array(["new"]).reshape(-1,1))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

data = pd.DataFrame({
    'city': ['Delhi', 'Mumbai', 'Pune', 'Delhi', 'Goa', 'Goa', 'Goa', 'Goa', 
              'Chennai', 'Chennai', np.nan, 'Delhi']
})

encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    min_frequency=2,
    max_categories=4,
    encoded_missing_value=-99
)

encoded = encoder.fit_transform(data[['city']])
print(encoded)
print(encoder.categories_)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import sklearn.datasets
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.model_selection import train_test_split
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
#from sklearn.datasets import fetch_california_housing
from sklearn.datasets import load_diabetes

# Instead of downloading, load as a DataFrame directly
#housing = fetch_california_housing(as_frame=True)
#X = housing.data
#y = housing.target
from sklearn.neighbors import KNeighborsRegressor 
from sklearn.preprocessing import StandardScaler
X,y =load_diabetes (return_X_y =True)
#from sklearn.datasets import preprocessor 
scalar=StandardScaler()
# Always assert this before fitting
assert X.shape[0] == y.shape[0], "Row count mismatch between X and y!"
X_scaled=scalar.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# --- Step 3: Cross validation (before tuning) ---
mod = PolynomialTransformer()
cv_scores = cross_val_score(mod, X_train, y_train, cv=5, scoring='r2')

print("Cross Validation R2 scores:", cv_scores.round(3))
print("Mean R2:", cv_scores.mean().round(3))
print("Std Dev:", cv_scores.std().round(3))

# --- Step 4: GridSearchCV — find best params ---
param_grid = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}
#mod = KNeighborsRegressor()
#read_model 
mod_grid= GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    cv=4,
    scoring='r2',
    verbose=1
)

predict_model= mod_grid.fit(X_train,y_train)
best_model = mod_grid.best_estimator_
predictions = best_model.predict(X_test)
print("\nFirst 5 predictions:", predictions[:5].round(1))
print("First 5 actual:     ", y_test[:5])

# --- Step 6: Final test score ---
test_score = best_model.score(X_test, y_test)
pd.DataFrame(test_score)
print(f"\nFinal Test R2 Score: {round(test_score, 3)}")
plt.scatter(predictions,y_test)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')